# DeepDEM paper figures

Compiled figure code for the DeepDEM manuscript (Mt. Baker site).

Section | Paper item
--- | ---
1 | Fig. 1 site context
2 | Fig. 5 residual maps
3 | Fig. 6 sample tiles (DSM)
4 | Fig. 7a land-cover error statistics, Fig. 7b SSIM histograms
5 | Table 3 patch-size experiment
6 | Fig. 8 and Table 4 transfer experiments
7 | Fig. 9 and Table 5 DTM prediction

In [ ]:
import functools, gc, json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib import colormaps
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

import rasterio
import rasterio.windows as window
from rasterio.plot import show
from rasterio.transform import rowcol
import rioxarray as rxr
import geopandas as gpd
import contextily as ctx
import seaborn as sns
from shapely import box
from shapely.geometry import Polygon
from skimage.metrics import structural_similarity as ssim

# data / statistics helpers (lib/figure_utils.py)
for _p in (Path.cwd().parent / 'lib', Path.cwd() / 'lib'):
    if (_p / 'figure_utils.py').exists():
        sys.path.insert(0, str(_p))
        break

import figure_utils as fu
compute_residuals = fu.compute_residuals     # used by the plotting cells

## Configuration

Everything site- or machine-specific is here. To run another site (e.g. Colorado) add an entry to `SITES`
and change `SITE`.

In [ ]:
SITE = 'baker_2015'

_BAKER = '/mnt/1.0_TB_VOLUME/ayushg12/baker_datasets/WV01_20150911_1020010042D39D00_1020010043455300'

SITES = {
    'baker_2015': dict(
        # --- data locations
        processed_dir=f'{_BAKER}/processed_rasters',           # final_*.tif training-ready rasters
        worldcover_path=f'{_BAKER}/tmp/final_worldcover_map.tif',   # used if it exists, otherwise downloaded and saved here
        outputs_dir='/mnt/Backups/ayushg12/DeepDEM_outputs',   # model inference rasters
        fig_dir='fig_dump',                                    # where figures are written
        # --- split used for training/validation (fraction of the x-extent used for training)
        train_split=0.65,
        # --- inference registry: model label -> file stem; files are <prefix><stem>.tif
        #     (+ _hs_multi.tif hillshade, + _ssim_map.tif SSIM map where available)
        inference_prefix='experiment_group_1_version_001_Baker_',
        inference=dict(
            unet='lidar_tools_unet',
            baseline='lidar_tools_256_bands_exp',        # baseline DeepDEM (stereo images + DSM + valid mask)
            ndvi='256_w_ndvi',
            tri='256_w_tri',
            ndvi_tri='lidar_tools_256',
            no_coreg='traditional_256',                  # alternative candidate: '256_bands_nocoreg'
            wv2013='WV02_20130911',
            wv2017='WV01_20170902',
            co='COwts',                                  # trained on Colorado, applied to Baker
            baker_co='BakerCO',                          # trained on Baker + Colorado
            lt256='lidar_tools_256',                     # 256-px model of the patch-size / transfer experiments (Tables 3-4, Fig. 8)
            p128='lidar_tools_128',                      # patch-size study (Table 3); 256 px = baseline
            p512='lidar_tools_512',
            dtm='DTM',                                   # DeepDEM trained on the lidar DTM (Fig. 9, Table 5)
        ),
        # SSIM-map stems differ for the NDVI + tri model (Fig. 7b)
        ssim_stem_override=dict(ndvi_tri='256_wv_all'),
        # --- tile windows (upper-left x, upper-left y in map units, tile size in pixels), see Figs. 6, 8, 9
        windows=dict(
            sparse=dict(xmin=591892.3, ymax=5405851.7, size=512, label='1) Bare/Sparse Vegetation ', xlims=[-3, 3]),
            snow=dict(xmin=589401.8, ymax=5407962.9, size=1024, label='2) Snow and Ice                ', xlims=[-2, 2]),
            dense=dict(xmin=590242.9, ymax=5397910.4, size=512, label='3) Dense Vegetation         ', xlims=[-16, 40]),
        ),
        # --- tile windows for the transfer figure (Fig. 8): same sparse / dense sites, 256-px tiles
        #     (the middle 'snow' entry keeps the list index layout of the figure cells but is not plotted)
        windows_transfer=dict(
            sparse=dict(xmin=591892.3, ymax=5405851.7, size=256),
            snow=dict(xmin=589401.8, ymax=5407962.9, size=1024),
            dense=dict(xmin=590242.9, ymax=5395410.4, size=256),
        ),
        # --- Fig. 7a axis limits per WorldCover class (10 dense veg., 30 sparse veg., 70 snow/ice)
        boxplot_ylims={10: (-10, 17), 30: (-1.65, 2.95), 70: (-1.6, 2.4), 'all': (-3.5, 4.8)},
        boxplot_yoffset={10: 1.4, 30: 0.26, 70: 0.2, 'all': 0.42},
    ),
    # Colorado (WV01 2019). Site-context cells (Fig. 1 / Fig. 2) and the lidar / stereo statistics only need the
    # processed rasters; tile windows and inference files for the small-window figures are not defined here.
    'co_2019': dict(
        processed_dir='/mnt/1.0_TB_VOLUME/ayushg12/CO_dataset/WV01WV01_20190930_102001008EC5AC00_102001008BE9BB00/processed_rasters',
        worldcover_path='/mnt/1.0_TB_VOLUME/ayushg12/CO_dataset/WV01WV01_20190930_102001008EC5AC00_102001008BE9BB00/processed_rasters/final_worldcover_map.tif',   # downloaded and saved here on first use
        outputs_dir='/mnt/Backups/ayushg12/DeepDEM_outputs',
        fig_dir='fig_dump',
        train_split=0.65,
        inference_prefix='experiment_group_1_version_001_',
        inference={},
        ssim_stem_override={},
        windows={},
        windows_transfer={},
        boxplot_ylims={}, boxplot_yoffset={},
    ),
}

cfg = SITES[SITE]

In [ ]:
# ---- derived paths and the variable names the figure cells below expect -------------------------------------
input_datapath = input_data_path = Path(cfg['processed_dir'])
output_datapath = output_data_path = Path(cfg['outputs_dir'])
out_figpath = Path(cfg['fig_dir']); out_figpath.mkdir(exist_ok=True)
train_split = cfg['train_split']

# ESA WorldCover on the lidar grid: use the file if it exists, otherwise download it once and save it there
worldcover_path = Path(cfg['worldcover_path'])
fu.ensure_worldcover(input_datapath / 'final_lidar_data.tif', worldcover_path)

nodata_mask_path = 'final_nodata_mask.tif'
lidar_data_path = 'final_lidar_data.tif'
lidar_hs_data_path = 'final_lidar_data_hs_multi.tif'


def inf_stem(key, ssim_map=False):
    stem = cfg['ssim_stem_override'].get(key, cfg['inference'][key]) if ssim_map else cfg['inference'][key]
    return cfg['inference_prefix'] + stem


def inf_file(key):
    return inf_stem(key) + '.tif'


def inf_ssim_file(key):
    return inf_stem(key, ssim_map=True) + '_ssim_map.tif'


# tile windows as the parallel lists used by the figure cells
_w = list(cfg['windows'].values())
xmin_list = [w['xmin'] for w in _w]
ymin_list = [w['ymax'] for w in _w]          # NB: upper-left y, consumed by get_bounds() below
chipsize = [w['size'] for w in _w]

## 1. Study site (Fig. 1)

In [ ]:
with rasterio.open(input_datapath / 'final_lidar_data.tif') as ds:
    lidar_img = ds.read(1)
    transform=ds.transform
    bounds = ds.bounds
    crs = ds.crs
    lidar_ndata = ds.nodata

mask = np.where(lidar_img == lidar_ndata, 0, 1)

lidar_hillshade_path = input_datapath / 'final_lidar_data_hs.tif'

with rasterio.open(lidar_hillshade_path) as ds:
    lidar_img_hs = ds.read(1)

lidar_img = np.ma.masked_where(mask==0, lidar_img)
lidar_img_hs = np.ma.masked_where(mask==0, lidar_img_hs)

In [ ]:
bbox_geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
roi_gdf = gpd.GeoDataFrame({"id": [1]}, geometry=[bbox_geom], crs=crs).to_crs(4326)
roi_gdf_buffer = roi_gdf.buffer(1)
train_extent = bounds.left + (bounds.right - bounds.left) * train_split
bbox_train = box(bounds.left, bounds.bottom, train_extent, bounds.top)
bbox_val = box(train_extent, bounds.bottom, bounds.right, bounds.top)
roi_train_gdf = gpd.GeoDataFrame({"id": [1]}, geometry=[bbox_train], crs=crs).to_crs(4326)
roi_val_gdf = gpd.GeoDataFrame({"id": [1]}, geometry=[bbox_val], crs=crs).to_crs(4326)

In [ ]:
# NDVI (HLS composite) used in Fig. 1b
with rasterio.open(input_datapath / 'final_ndvi.tif') as ds:
    ndvi_img = ds.read(1)

In [ ]:
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- 2. Load world boundaries
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")

# --- 3. Create orthographic projection (centered roughly on the U.S.)
fig = plt.figure(figsize=(6, 6))
ax = plt.axes(projection=ccrs.Orthographic(central_longitude=-100, central_latitude=40))

# --- 4. Add land, ocean, and borders
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.OCEAN, facecolor="white")
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, color="black")
ax.add_feature(cfeature.BORDERS, linewidth=0.5, color="black")

# --- 5. Plot country outlines
world.plot(ax=ax, transform=ccrs.PlateCarree(),
           edgecolor="black", facecolor="none", linewidth=0.3)

# --- 6. Plot bounding box (black outline, filled white)
centroid = roi_gdf.geometry.iloc[0].centroid
lon, lat = centroid.x, centroid.y
ax.plot(lon, lat, marker='*', color='red', markersize=14,
        transform=ccrs.PlateCarree(), zorder=4)

# --- 7. Make it circular
ax.set_global()
circle = plt.Circle((0.5, 0.5), 0.5, transform=ax.transAxes,
                    color='none', ec='none', clip_on=False)
ax.add_patch(circle)
ax.set_clip_path(circle)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx
import geopandas as gpd
import pyproj
import numpy as np
import matplotlib.patches as mpatches
from imview import pltlib

fig, ax = plt.subplots(1, 2, figsize=(19, 10), width_ratios=[1.05, 1])

# Plot regions of interest
roi_train_gdf.plot(facecolor="none", ax=ax[0], label='Training Region', edgecolor="blue", linewidth=2, linestyle='-')
roi_val_gdf.plot(facecolor="none", ax=ax[0], label='Validation Region', edgecolor="red", linewidth=2, linestyle='-')
train_patch = mpatches.Patch(edgecolor="blue", facecolor="none", label="Training Region")
val_patch = mpatches.Patch(edgecolor="red", facecolor="none", label="Validation Region")

ax[0].legend(handles=[train_patch, val_patch], loc="lower right", fontsize=14)

# --- Get and expand bounds (zoom out) ---
xmin, ymin, xmax, ymax = roi_gdf.total_bounds
zoom_factor = 1.20  # >1 zooms out; try 1.5 or 2 for wider view
x_center, y_center = (xmin + xmax) / 2, (ymin + ymax) / 2
x_width, y_height = (xmax - xmin) * zoom_factor, (ymax - ymin) * zoom_factor

ax[0].set_xlim(x_center - x_width / 2 + 0.02, x_center + x_width / 2 + 0.06)
ax[0].set_ylim(y_center - y_height / 2, y_center + y_height / 2)

# Add basemap (in Web Mercator)
ctx.add_basemap(ax[0], crs=roi_train_gdf.crs.to_string(), source=ctx.providers.Esri.WorldImagery, attribution=False)

# --- Convert axis tick labels to lat/lon ---
# Get limits in projected coordinates (Web Mercator)
xmin, xmax = ax[0].get_xlim()
ymin, ymax = ax[0].get_ylim()
lon_labels = np.arange(xmin+0.02, xmax, 0.075)
lat_labels = np.arange(ymin+0.02, ymax, 0.075)

# Apply ticks and labels
ax[0].set_xticks(lon_labels)
ax[0].set_xticklabels([f"  {lon:.2f}°W" for lon in lon_labels])
ax[0].set_yticks(lat_labels)
ax[0].set_yticklabels([f"{lat:.2f}°N" for lat in lat_labels])
ax[0].tick_params(axis='y', labelrotation=90)
ylabels = ax[0].yaxis.get_ticklabels()

# Iterate through the labels and set their vertical alignment
for label in ylabels:
    label.set_verticalalignment('center')

# Styling
ax[0].tick_params(axis='both', which='major', labelsize=16)

ax[0].text(
    0.02, 0.98,                      # position (x, y) in axes fraction coords
    "a",                             # text
    transform=ax[0].transAxes,
    fontsize=20, fontweight="bold",
    va="top", ha="left",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        boxstyle="square,pad=0.3",
        alpha=0.5
    )
)

ax[0].set_title("RGB Basemap", fontsize=24)

# im = ax[1].imshow(ndvi_img, cmap='RdYlGn', vmin=-1, vmax=1)
# fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04, label='NDVI Value')
pltlib.iv(ndvi_img,ax=ax[1],clim=(-1, 1), cmap='RdYlGn',label='NDVI',scalebar=True)
# ax[1].set(xticks=[], yticks=[])
ax[1].text(
    0.02, 0.98,                      # position (x, y) in axes fraction coords
    "b",                             # text
    transform=ax[1].transAxes,
    fontsize=20, fontweight="bold",
    va="top", ha="left",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        boxstyle="square,pad=0.3",
        alpha=0.5
    )
)
scalebar = AnchoredSizeBar(ax[1].transData,
                        2000 , '2 km', 'lower right', 
                        color='black', frameon=True,
                        pad = 0.35, sep=5,
                        fontproperties = {'weight':'normal', 'size':15},
                        size_vertical=50)

ax[1].add_artist(scalebar)
cbar = ax[1].images[0].colorbar
cbar.ax.tick_params(labelsize=14)
cbar.set_label('NDVI', size=18) 

ax[1].set_title("NDVI", fontsize=24)

plt.tight_layout()

plt.savefig(out_figpath / "fig_roi_ndvi.png", dpi=300, bbox_inches='tight')

In [ ]:
import rioxarray as rxr
import pyproj
import easysnowdata
from shapely.ops import transform
from shapely.geometry import box


baker_dsm = rxr.open_rasterio(input_datapath / 'final_lidar_data.tif').squeeze()
# baker_dsm = rxr.open_rasterio('/mnt/1.0_TB_VOLUME/ayushg12/CO_dataset/CO_3DEP/CO_3DEP-DSM_mos-temp.tif').squeeze()

# Getting initial common region
xmin, ymin, xmax, ymax = baker_dsm.rio.bounds()
roi = box(xmin, ymin, xmax, ymax)

source_crs = pyproj.CRS(baker_dsm.rio.crs)
destination_crs = pyproj.CRS("EPSG:4326")
transformer = pyproj.Transformer.from_crs(source_crs, destination_crs, always_xy=True)
# 4. Use shapely.ops.transform to apply the transformation
roi_4326 = transform(transformer.transform, roi)

land_classification_match = fu.load_worldcover(worldcover_path, baker_dsm)

valid_cls = [10, 30, 60, 70, 100]
mask = land_classification_match.isin(valid_cls)
land_classification_match = land_classification_match.where(mask, 0)

land_classification_match = land_classification_match.where(~((land_classification_match==30) | 
                                                                (land_classification_match==60) | 
                                                                (land_classification_match==100)), 30)

# new_classification = {}
valid_cls = [10, 30, 70]
class_attr = land_classification_match.attrs['class_info']
new_class_attr = {k: class_attr[k] for k in valid_cls}
new_class_attr[30]['name'] = 'Bare/Sparse \nvegetation'
new_class_attr[0] = {'name': 'Other', 'color': '#000000'}

cmap = plt.cm.colors.ListedColormap(
            [new_class_attr[key]["color"] for key in sorted(new_class_attr.keys())]
        )

In [ ]:
import matplotlib

# fig, ax = plt.subplots(1, 2, figsize=(12.8, 10), width_ratios=[1, 1])
fig, ax = plt.subplots(1, 2, figsize=(19, 10), width_ratios=[1, 1])


cmap_ele = colormaps.get_cmap('viridis')
cmap_ele.set_bad(color='black')

asp_dsm_flat = lidar_img.flatten()
print(asp_dsm_flat.min(), asp_dsm_flat.max())
vmin, vmax = asp_dsm_flat.min()+200, np.percentile(asp_dsm_flat[asp_dsm_flat!=-9999], 99)
# np.percentile(asp_dsm_flat[asp_dsm_flat!=-9999], 5), np.percentile(asp_dsm_flat[asp_dsm_flat!=-9999], 99)
print(vmin, vmax)
im = ax[0].imshow(lidar_img, cmap=cmap_ele, vmin=vmin, vmax=vmax, interpolation=None)
ax[0].imshow(lidar_img_hs, cmap='gray', alpha=0.5, interpolation=None)
# show(asp_dsm.data, ax=ax[2], transform=transform, vmin=vmin, vmax=vmax, interpolation=None)
# show(asp_dsm_hs.data, ax=ax[2], transform=transform, cmap='gray', alpha=0.5, interpolation=None)

cax = ax[0].inset_axes([0.03, 0.05, 0.4, 0.02])
cax.xaxis.set_ticks_position("bottom")
ax[0].add_patch(Rectangle((0.01, 0.01), 0.45, 0.07, transform=ax[0].transAxes, facecolor="white", alpha=0.7))
cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
cbar.ax.tick_params(labelsize=12)
# cbar.set_label('Elevation (meters)', size=14)
# ax[0].set_title('LIDAR DSM', fontsize=17)
ax[0].set_title('LIDAR DSM', fontsize=23)
ax[0].set_xticks([])
ax[0].set_yticks([])

ax[0].text(
    0.02, 0.98,                      # position (x, y) in axes fraction coords
    "c",                             # text
    transform=ax[0].transAxes,
    fontsize=20, fontweight="bold",
    va="top", ha="left",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        boxstyle="square,pad=0.3",
        alpha=0.5
    )
)


class_values = sorted(new_class_attr.keys())
bounds = [
            (class_values[i] + class_values[i + 1]) / 2 for i in range(len(class_values) - 1)
        ]
bounds = [class_values[0] - 0.5] + bounds + [class_values[-1] + 0.5]

norm = matplotlib.colors.BoundaryNorm(bounds, cmap.N)

land_classification_match.plot.imshow(ax=ax[1], cmap=cmap, norm=norm, add_colorbar=False)

legend_handles = []
class_names = []
for class_value, class_info in new_class_attr.items():
    legend_handles.append(
        plt.Rectangle((0, 0), 1, 1, facecolor=class_info["color"], edgecolor="black")
    )
    class_names.append(class_info["name"])

legend_kwargs = {}
default_legend_kwargs = {
    "bbox_to_anchor": (1.005, 0.5),  # right of the axes
    "loc": "center left",
    "ncol": 1,                      # vertical legend
    "frameon": False,
    "fontsize":16,
    "handlelength": 1.5,
    "handleheight": 1.5,
}
legend_kwargs = {**default_legend_kwargs, **legend_kwargs}


ax[1].legend(legend_handles, class_names, **legend_kwargs)

ax[1].set_aspect("equal")
ax[1].set_xlabel("")
ax[1].set_ylabel("")
ax[1].set(xticks=[], yticks=[])
# ax[1].set_title("ESA WorldCover (2021)", fontsize=17)
ax[1].set_title("ESA WorldCover (2021)", fontsize=23)
# ax[1].set_title("")

ax[1].text(
    0.02, 0.98,                      # position (x, y) in axes fraction coords
    "d",                             # text
    transform=ax[1].transAxes,
    fontsize=20, fontweight="bold",
    va="top", ha="left",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        boxstyle="square,pad=0.3",
        alpha=0.5
    )
)

plt.tight_layout()

## 2. Elevation residual maps (Fig. 5)

In [ ]:
# raw lidar DSM (nodata kept as -9999), stereo DSM + hillshade, baseline DeepDEM DSM + hillshade on the stereo grid
with rasterio.open(input_datapath / 'final_lidar_data.tif') as ds:
    lidar_img = ds.read(1)
    lidar_ndata = ds.nodata
with rasterio.open(input_datapath / 'final_lidar_data_hs.tif') as ds:
    lidar_img_hs = ds.read(1)
with rasterio.open(input_datapath / 'final_asp_dsm.tif') as ds:
    asp_dsm_img = ds.read(1)
with rasterio.open(input_datapath / 'final_asp_dsm_hs.tif') as ds:
    asp_dsm_hs = ds.read(1)

asp_img_ds = rxr.open_rasterio(input_datapath / 'final_asp_dsm.tif')
deepdem_img_ds = rxr.open_rasterio(output_datapath / inf_file('baseline'))
deepdem_img_hs_ds = rxr.open_rasterio(output_datapath / inf_file('baseline').replace('.tif', '_hs_multi.tif'))
deepdem_img_ds = deepdem_img_ds.rio.reproject_match(asp_img_ds).squeeze().values
deepdem_img_hs_ds = deepdem_img_hs_ds.rio.reproject_match(asp_img_ds).squeeze().values

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(24, 8), width_ratios=[1, 1, 1.06])

# LiDAR DSM plot
asp_dsm_flat = lidar_img.flatten()
vmin, vmax = np.percentile(asp_dsm_flat[asp_dsm_flat!=-9999], 1), np.percentile(asp_dsm_flat[asp_dsm_flat!=-9999], 99)
im = ax[0].imshow(lidar_img, vmin=vmin, vmax=vmax, interpolation=None)
ax[0].imshow(lidar_img_hs, cmap='gray', alpha=0.5, interpolation=None)

cax = ax[0].inset_axes([0.03, 0.07, 0.4, 0.03])
cax.xaxis.set_ticks_position("bottom")
ax[0].add_patch(Rectangle((0.01, 0.01), 0.45, 0.1, transform=ax[0].transAxes, facecolor="white", alpha=0.7))
cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
cbar.ax.tick_params(labelsize=15)
# cbar.set_label('Elevation (meters)', size=14)
ax[0].set_title('LIDAR DSM', fontsize=22)
ax[0].set(xticks=[], yticks=[])

# ASP Residual plot
resid_cmap = colormaps.get_cmap('RdBu_r')
resid_cmap.set_bad(color='black')

vmin, vmax = -6, 6
mask = (lidar_img == lidar_ndata) | (asp_dsm_img == 0)
asp_residual = asp_dsm_img - lidar_img
asp_dsm_hs_masked = np.ma.masked_where(mask==1, asp_dsm_hs)
asp_residual[mask==1] = np.nan
im = ax[1].imshow(asp_residual, cmap=resid_cmap, vmin=vmin, vmax=vmax, interpolation=None)
ax[1].imshow(asp_dsm_hs_masked, cmap='gray', alpha=0.5, interpolation=None)
ax[1].set(xticks=[], yticks=[])
ax[1].set_title('Co-registered stereo DSM residual', fontsize=22)
scalebar = AnchoredSizeBar(ax[1].transData,
                        2000 , '2 km', 'lower right', 
                        color='black', frameon=True,
                        pad = 0.35, sep=5,
                        fontproperties = {'weight':'normal', 'size':17},
                        size_vertical=50)

ax[1].add_artist(scalebar)

# DeepDEM Residual plot
deepdem_residual = deepdem_img_ds - lidar_img
deepdem_img_hs_ds_masked = np.ma.masked_where(mask==1, deepdem_img_hs_ds)
deepdem_residual[mask==1] = np.nan
im = ax[2].imshow(deepdem_residual, cmap=resid_cmap, vmin=vmin, vmax=vmax, interpolation=None)
ax[2].imshow(deepdem_img_hs_ds_masked, cmap='gray', alpha=0.5, interpolation=None)
cbar = fig.colorbar(im, ax=ax[2], fraction=0.045, pad=0.01, label='Elevation Residual (meters)')
ax[2].set(xticks=[], yticks=[])
ax[2].set_title('Baseline DeepDEM DSM residual', fontsize=22);
cbar.ax.tick_params(labelsize=16)
cbar.set_label('Stereo - LiDAR DSM [m]', size=18) 

plt.tight_layout()
plt.savefig(out_figpath / "fig_dsm_residuals.png", dpi=300, bbox_inches='tight')

## 3. Sample tiles over three land-cover types (Fig. 6)

In [ ]:
# tile reader (lib/figure_utils.py); model slots: Baker 2013, Baker 2017, baseline (shown in Fig. 6), Colorado -> Baker
read_comparison_files = functools.partial(
    fu.read_comparison_files, input_data_path=input_data_path, output_data_path=output_data_path,
    models=[inf_file(k) for k in ['wv2013', 'wv2017', 'baseline', 'co']],
    lidar_data_path=lidar_data_path, lidar_hs_data_path=lidar_hs_data_path, nodata_mask_path=nodata_mask_path)

In [ ]:
def plot_dsm_visual(all_data, chipsize, fig, ax, xlims=[-8, 8], no_label='1', label_side='left'):
    
    asp_dsm_img, asp_dsm_img_hs, _, _, _, _, model_inference_img, model_inference_img_hs, _, _, lidar_dsm_img, lidar_dsm_img_hs, nodata_img, ortho_channel1_img = all_data
    
    # Compute vmin and vmax for DSM plots
    lidar_dsm_img_flat = lidar_dsm_img.flatten()
    vmin, vmax = np.percentile(lidar_dsm_img_flat[lidar_dsm_img_flat!=-9999], 1), np.percentile(lidar_dsm_img_flat[lidar_dsm_img_flat!=-9999], 99)
    
    # Plotting DSMs
    cmap = colormaps.get_cmap('viridis')
    cmap.set_bad(color='dimgrey')
    
    show(ortho_channel1_img, ax=ax[0], )
    ax[0].imshow(ortho_channel1_img, cmap='gray')

    im = ax[1].imshow(asp_dsm_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[1].imshow(asp_dsm_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[2].imshow(model_inference_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[2].imshow(model_inference_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[3].imshow(lidar_dsm_img , vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[3].imshow(lidar_dsm_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    refined_residuals = compute_residuals(model_inference_img, lidar_dsm_img, nodata_img)
    original_residuals = compute_residuals(asp_dsm_img, lidar_dsm_img, nodata_img)
    refined_residuals = refined_residuals[(refined_residuals<xlims[1])&(refined_residuals>xlims[0])]
    original_residuals = original_residuals[(original_residuals<xlims[1])&(original_residuals>xlims[0])]
    
    _, ssim_asp_map = ssim(lidar_dsm_img_hs, asp_dsm_img_hs.data, data_range=asp_dsm_img_hs.max() - asp_dsm_img_hs.min(), win_size=15, full=True)
    _, ssim_deepdem_map = ssim(lidar_dsm_img_hs, model_inference_img_hs, data_range=model_inference_img_hs.max() - model_inference_img_hs.min(), win_size=15, full=True)
    
    ssim_asp = np.ma.masked_where(nodata_img==0, ssim_asp_map)
    ssim_deepdem = np.ma.masked_where(nodata_img==0, ssim_deepdem_map)
    print(f'MAE initial stereo: {np.ma.abs(original_residuals).mean():.2f}, MAE DeepDEM: {np.ma.abs(refined_residuals).mean():.2f}')
    print(f'SSIM initial stereo: {ssim_asp.mean():.3f}, SSIM DeepDEM: {ssim_deepdem.mean():.3f}')

    
    ax[4].hist(refined_residuals, bins=200, alpha=0.4, label='DeepDEM', color='C1');
    ax[4].hist(original_residuals, bins=200, alpha=0.5, label='Stereo', color='C0');

    ax[4].axvline(x=0, color='k', linestyle='--')
    pos = ax[4].get_position()
    ax[4].tick_params(axis='both', labelsize=9)
    ax[4].set_xlim(xlims)
    ax[4].legend(loc=f'upper {label_side}', fontsize=9)
    
    textstr = f"MAE stereo: {np.ma.abs(original_residuals).mean():.3f}\nMAE DeepDEM: {np.ma.abs(refined_residuals).mean():.3f}\nSSIM stereo: {ssim_asp.mean():.3f}\nSSIM DeepDEM: {ssim_deepdem.mean():.3f}"
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    if label_side=='left':
        ax[4].text(0.04, 0.77, textstr, fontsize=7.5, transform=ax[4].transAxes, verticalalignment='top', bbox=props)
    else:
        ax[4].text(0.52, 0.77, textstr, fontsize=7.5, transform=ax[4].transAxes, verticalalignment='top', bbox=props)
    # Adding scalebars
    scale = chipsize/512
    scalebar = AnchoredSizeBar(ax[0].transData,
                        100*scale , f'{int(100*scale)} meters', 'lower right', size_vertical=10*scale,
                        color='white', frameon=False, pad = 0.25, sep=5,
                        fontproperties = {'weight':'semibold', 'size':10})

    ax[0].add_artist(scalebar)

    # Add scence no. label on ortho image plot
    ax[0].add_patch(Rectangle((0.01, 0.86), 0.85, 0.13, transform=ax[0].transAxes, facecolor="white", alpha=0.8))
    ax[0].text(0.43, 0.92, no_label,
            transform=ax[0].transAxes,  # interpret as fraction of axes
            fontsize=11,
            color='black',
            ha='center', va='center')  # horizontal and vertical alignment

    # Add colorbar for the ASP DSM
    cax = ax[1].inset_axes([0.05, 0.1, 0.4, 0.03])
    cax.xaxis.set_ticks_position("bottom")
    ax[1].add_patch(Rectangle((0.015, 0.01), 0.53, 0.15, transform=ax[1].transAxes, facecolor="white", alpha=0.7))
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=10)
    
    # Shifting histogram axes
    ax[4].set_position([pos.x0 + 0.026, pos.y0 + 0.021, pos.width * 0.94, pos.height * 0.91])
    
    return ssim_asp, ssim_deepdem

In [ ]:
fig, ax = plt.subplots(3, 5, figsize = (17.8, 10.5))
plt.subplots_adjust(wspace=0.01, hspace=0.04)

all_data1 = read_comparison_files(xmin_list, ymin_list, chipsize, 0)
ssim_asp1, ssim_deepdem1 = plot_dsm_visual(all_data1, chipsize[0], fig, ax[0], xlims=[-3, 3], no_label='1) Bare/Sparse Vegetation ')
all_data2 = read_comparison_files(xmin_list, ymin_list, chipsize, 1)
ssim_asp2, ssim_deepdem2 = plot_dsm_visual(all_data2, chipsize[1], fig, ax[1], xlims=[-2, 2], no_label='2) Snow and Ice                ')
all_data3 = read_comparison_files(xmin_list, ymin_list, chipsize, 2)
ssim_asp3, ssim_deepdem3 = plot_dsm_visual(all_data3, chipsize[2], fig, ax[2], xlims=[-16, 40], no_label='3) Dense Vegetation         ', label_side='right')

ax[0, 0].set_title('Reference ortho image', fontsize=13)
ax[0, 1].set_title('Co-registered stereo DSM', fontsize=13)
ax[0, 2].set_title('DeepDEM DSM', fontsize=13)
ax[0, 3].set_title('3DEP LiDAR DSM', fontsize=13)
ax[0, 4].set_title('Residual histograms [m]', fontsize=13)

# Turn off tickmarks for images
for ax_ in ax[:, 0:4].flat:
    ax_.set_xticklabels('')
    ax_.set_yticklabels('')
    ax_.set(xticks=[], yticks=[])
    
plt.savefig(out_figpath / 'baker_dsm_comparison.png', dpi=300, bbox_inches='tight')

## 4. Land-cover error statistics and SSIM (Table S1, Fig. 7)

In [ ]:
# free the large rasters used by the figures above (the statistics below need several tens of GB)
import gc
for _n in ['lidar_img', 'lidar_img_hs', 'asp_dsm_img', 'asp_dsm_hs', 'asp_img_ds', 'deepdem_img_ds', 'deepdem_img_hs_ds',
           'asp_residual', 'deepdem_residual', 'ndvi_img', 'land_classification', 'land_classification_match', 'mask',
           'all_data1', 'all_data2', 'all_data3']:
    globals().pop(_n, None)
gc.collect()

In [ ]:
asp_path = input_data_path / 'final_asp_dsm.tif'
file_list = [asp_path] + [output_data_path / inf_file(k) for k in ['unet', 'no_coreg', 'ndvi_tri', 'ndvi', 'tri', 'baseline']]
labels = ['ASP', 'UNet', 'DeepDEM no coreg', 'DeepDEM w ancillary', 'DeepDEM w NDVI', 'DeepDEM w tri', 'DeepDEM']
df_val, _, class_info, data_dict_val, _ = fu.compute_DSM_stats(file_list, labels, input_data_path, train_split, worldcover_path,
                                                              compute_train=False, ssim_maps=False)  # train-region stats: see supplement

In [ ]:
df_val

In [ ]:
ylims, yoffset = cfg['boxplot_ylims'], cfg['boxplot_yoffset']
dict_labels = ['ASP', 'DeepDEM', 'UNet', 'DeepDEM no coreg', 'DeepDEM w NDVI', 'DeepDEM w tri', 'DeepDEM w ancillary']
labels = [ 'Co-registered\nstereo', 'Baseline\nDeepDEM', 'UNet', 'DeepDEM\nno coreg', 'DeepDEM with\n only NDVI', 'DeepDEM with\n only tri. error', 'DeepDEM with\n NDVI + tri. error']
colors = ['green', '#56B4E9', 'red', 'orchid', 'blue', 'peachpuff', 'orange']
# colors = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00']
# colors = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#FF0000', 'orchid']

# colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2']

# lulc_classes = {10: 'Dense vegetation', 30: 'Sparse vegetation', 70: 'Snow and Ice', 'all': 'Overall'}
lulc_classes = {'all': 'Overall', 10: 'Dense vegetation', 30: 'Sparse vegetation', 70: 'Snow and Ice'}
fig, axs = plt.subplots(1, 4, figsize = (18, 5))
plt.style.use('ggplot')
medianprops = dict(color="black",linewidth=1.5, linestyle='-.')

np.random.seed(42)

for ax, cls in zip(axs.flatten(), lulc_classes.keys()):
    boxplot_data = []
    for i, label in enumerate(dict_labels):
        dsm_data, lidar_data = data_dict_val[cls][label]
        if i!=0:
            mae_val = np.mean(np.abs(residuals))
        residuals = dsm_data - lidar_data
        boxplot_data.append(residuals)
        
    mae_values = [np.mean(np.abs(d)) for d in boxplot_data]
    
    ax.set_ylabel("Residual [m]", fontsize=20)
    im = ax.boxplot(boxplot_data, patch_artist=True, tick_labels=labels, sym='', medianprops=medianprops) #, whis=(5, 95))
    ax.set_ylim(*ylims[cls])

    for patch, color in zip(im['boxes'], colors):
        patch.set_facecolor(color)
        
    # --- Add translucent white strip at the top
    ymax = ax.get_ylim()[1]
    yoff = yoffset[cls]*1.1
    ystrip_bottom = ymax - yoff  # adjust height of the strip
    ax.axhspan(ystrip_bottom, ymax, color='white', alpha=0.75, zorder=2)
    ax.axhline(y=0, color='white', linestyle='-', linewidth=1, zorder=1)

    # --- Write MAE text in the strip
    for i, (x, mae) in enumerate(zip(range(1, len(labels)+1), mae_values)):
        ax.text(x, ymax - (yoff/2), f"{mae:.2f}",
                ha='center', va='center',
                fontsize=14,
                color='black', zorder=3)

    ax.grid(axis='x')
    ax.tick_params(axis='y', labelsize=14.7) 
    ax.tick_params(axis='x', labelsize=18.7) 
    print(len(lulc_classes[cls]), 1-(len(lulc_classes[cls])/17)*0.393)
    # ax.text(1-(len(lulc_classes[cls])/17)*0.23, 0.9, f"{lulc_classes[cls]}",
    #         transform=ax.transAxes,
    #         fontsize=27,
    #         verticalalignment='top')

# axs[0].set_ylabel('')
axs[1].set_ylabel('')
axs[2].set_ylabel('')
axs[3].set_ylabel('')
axs[0].set_xticks([])
axs[1].set_xticks([])
axs[2].set_xticks([])
axs[3].set_xticks([])

plt.tight_layout()
plt.savefig(out_figpath / 'baker_dsm_residual_boxplots.png', dpi=300, bbox_inches='tight')

### Fig. 7b: SSIM distributions

In [ ]:
lidar_path = input_datapath / 'final_lidar_data.tif'
nodata_path = input_datapath / 'final_nodata_mask.tif'

lidar_ds = rxr.open_rasterio(lidar_path).squeeze()
bbox = lidar_ds.rio.bounds()
minx, maxx = bbox[0], bbox[2]
miny, maxy = bbox[1], bbox[3]
train_extent = minx + (maxx - minx) * train_split
bbox_train = (minx, miny, train_extent, maxy)
bbox_val = (train_extent, miny, maxx, maxy)

# per-pixel SSIM maps (hillshade SSIM vs. lidar), one per model
asp_path = input_datapath / 'final_asp_dsm_ssim_map.tif'
unet_path = output_datapath / inf_ssim_file('unet')
deepdem_path = output_datapath / inf_ssim_file('baseline')
deepdem_ndvi_path = output_datapath / inf_ssim_file('ndvi')
deepdem_tri_path = output_datapath / inf_ssim_file('tri')
deepdem_anc_path = output_datapath / inf_ssim_file('ndvi_tri')
deepdem_nocoreg_path = output_datapath / inf_ssim_file('no_coreg')

In [ ]:
# Computing values for different models on validation set

cls = 'all'
asp_vals = fu.compute_vals(asp_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)
unet_vals = fu.compute_vals(unet_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)
deepdem_vals = fu.compute_vals(deepdem_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)
print('half done')
deepdem_ndvi_vals = fu.compute_vals(deepdem_ndvi_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)
deepdem_tri_vals = fu.compute_vals(deepdem_tri_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)
deepdem_anc_vals = fu.compute_vals(deepdem_anc_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)
deepdem_nocoreg_vals = fu.compute_vals(deepdem_nocoreg_path, nodata_path, worldcover_path, cls=cls, bbox=bbox_val)

In [ ]:
import seaborn as sns

data = {
    'Co-registered stereo': asp_vals,
    'Baseline DeepDEM': deepdem_vals,
    'UNet': unet_vals,
    'DeepDEM no coreg.': deepdem_nocoreg_vals,
    'DeepDEM with \nonly NDVI': deepdem_ndvi_vals,
    'DeepDEM with \nonly tri. error': deepdem_tri_vals,
    'DeepDEM with \nNDVI + tri. error': deepdem_anc_vals
}

colors = {
    'Co-registered stereo': 'green',
    'UNet': 'red',
    'DeepDEM no coreg.': 'orchid',
    'Baseline DeepDEM': '#56B4E9',
    'DeepDEM with \nonly NDVI': 'blue',
    'DeepDEM with \nonly tri. error': 'peachpuff',
    'DeepDEM with \nNDVI + tri. error': 'orange'
}

# colors_li = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#FF0000', 'orchid']
# dict_labels = ['ASP', 'DeepDEM', 'UNet', 'DeepDEM no coreg', 'DeepDEM w NDVI', 'DeepDEM w tri', 'DeepDEM w ancillary']
# ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b', '#e377c2']

# colors = {
#     'Co-registered stereo': '#1f77b4',
#     'UNet': '#d62728',
#     'DeepDEM no coreg.': '#2ca02c',
#     'Baseline DeepDEM': '#8c564b',
#     'DeepDEM with \nonly NDVI': '#9467bd',
#     'DeepDEM with \nonly tri. error': '#e377c2',
#     'DeepDEM with \nNDVI + tri. error': '#ff7f0e'
# }

# colors = {
#     'Co-registered stereo': '#E69F00',
#     'UNet': '#009E73',
#     'DeepDEM no coreg.': '#F0E442',
#     'Baseline DeepDEM': '#56B4E9',
#     'DeepDEM with \nonly NDVI': '#0072B2',
#     'DeepDEM with \nonly tri. error': '#FF0000',
#     'DeepDEM with \nNDVI + tri. error': 'orchid'
# }

plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(18, 4.5))

# --- Plot KDEs
for label, values in data.items():
    print(label)
    idxs = np.random.randint(0, len(values), size=100000)
    print(np.mean(values), np.median(values))
    sns.histplot(
        values[idxs],
        ax=ax,
        bins=150,
        fill=True,
        alpha=0.35,
        linewidth=0,
        label=label,
        color=colors[label],
    )
    
    # Mean line
    mean_val = np.mean(values)
    ax.axvline(mean_val, color=colors[label], linewidth=2)
    
# --- Legend and labels
leg = ax.legend(title="", loc="lower left", frameon=True, facecolor="darkgray", ncols=2, fontsize=15, edgecolor="black")
for lh in leg.legend_handles:
    lh.set_alpha(0.5)
    
ax.set_xlabel("Structural Similarity (SSIM)", fontsize=22)
ax.set_ylabel("Count", fontsize=22)
ax.tick_params(axis='both', labelsize=15)
ax.set_xlim(-0.1, 0.8)
plt.tight_layout()
plt.savefig(out_figpath / "baker_dsm_ssim.png", dpi=300, bbox_inches='tight')

## 5. Patch-size experiment (Table 3)

These tables also report mean SSIM, so `compute_DSM_stats` reads the per-model SSIM maps (`*_ssim_map.tif`).

In [ ]:
# free the large arrays from section 4
import gc
for _n in ['data_dict_val', 'data_dict_train', 'lidar_ds', 'asp_vals', 'unet_vals', 'deepdem_vals', 'deepdem_ndvi_vals',
           'deepdem_tri_vals', 'deepdem_anc_vals', 'deepdem_nocoreg_vals', 'noise', 'data', 'values']:
    globals().pop(_n, None)
gc.collect()

In [ ]:
file_list = [output_data_path / inf_file(k) for k in ['p128', 'lt256', 'p512']]
labels = ['128', '256', '512']
df_val, _, class_info, _, _ = fu.compute_DSM_stats(file_list, labels, input_data_path, train_split, worldcover_path, compute_train=False)
df_val

## 6. Temporal and spatial transfer (Fig. 8, Table 4)

Colour-shaded relief, residuals and hillshade SSIM for two tiles. The model columns are, in order: baseline
(Baker 2015 -> Baker 2015), Baker 2015 -> Baker 2013, Baker 2015 -> Baker 2017, Colorado 2019 -> Baker 2015.

In [ ]:
# tile windows for this figure
_w = list(cfg['windows_transfer'].values())
xmin_list = [w['xmin'] for w in _w]
ymin_list = [w['ymax'] for w in _w]
chipsize = [w['size'] for w in _w]

In [ ]:
# tile reader that also returns the SSIM maps (lib/figure_utils.py); columns: baseline, 2013, 2017, Colorado -> Baker
read_comparison_files = functools.partial(
    fu.read_comparison_files_ssim, input_data_path=input_data_path, output_data_path=output_data_path,
    models=[inf_file(k) for k in ['lt256', 'wv2013', 'wv2017', 'co']],
    lidar_data_path=lidar_data_path, nodata_mask_path=nodata_mask_path)

In [ ]:
# Plotting funs

def plot_dsm_axs(all_data, ax):

    asp_dsm_img, asp_dsm_img_hs, asp_dsm_img_ssim, unet_inference_img, unet_inference_img_hs, unet_inference_img_ssim, deepdem_inference128_img, deepdem_inference128_img_hs, deepdem_inference128_img_ssim, \
    deepdem_inference256_img, deepdem_inference256_img_hs, deepdem_inference256_img_ssim, deepdem_inference512_img, deepdem_inference512_img_hs, deepdem_inference512_img_ssim, lidar_dsm_img, lidar_dsm_img_hs, _, _, _ = all_data
    
    # Getting vmin/vmax for DSM plots
    lidar_dsm_img_flat = lidar_dsm_img.flatten()
    vmin, vmax = np.percentile(lidar_dsm_img_flat[lidar_dsm_img_flat!=-9999], 1), np.percentile(lidar_dsm_img_flat[lidar_dsm_img_flat!=-9999], 99)

    cmap = colormaps.get_cmap('viridis')
    cmap.set_bad(color='silver')

    im = ax[0].imshow(asp_dsm_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[0].imshow(asp_dsm_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[1].imshow(unet_inference_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[1].imshow(unet_inference_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[2].imshow(deepdem_inference128_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[2].imshow(deepdem_inference128_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[3].imshow(deepdem_inference256_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[3].imshow(deepdem_inference256_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[4].imshow(deepdem_inference512_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[4].imshow(deepdem_inference512_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    ax[5].imshow(lidar_dsm_img, vmin=vmin, vmax=vmax, cmap=cmap, interpolation=None)
    ax[5].imshow(lidar_dsm_img_hs, alpha=0.5, cmap='gray', interpolation=None)

    # Adding scalebars
    scalebar = AnchoredSizeBar(ax[0].transData,
                            50 , '50 meters', 'lower right', 
                            color='white',
                            frameon=False,
                            pad = 0.25,
                            sep=5,
                            fontproperties = {'weight':'semibold', 'size':10},
                            size_vertical=5)

    ax[0].add_artist(scalebar)
    ax[0].add_patch(Rectangle((0.68, 0.005), 0.315, 0.13, transform=ax[0].transAxes, facecolor="black", alpha=0.6))

    ax[0].set_ylabel('DSM [m]', fontsize=18)
    
    # Adding colorbars
    cax = ax[0].inset_axes([0.03, 0.1, 0.4, 0.03])
    cax.xaxis.set_ticks_position("bottom")
    ax[0].add_patch(Rectangle((0.01, 0.01), 0.48, 0.15, transform=ax[0].transAxes, facecolor="white", alpha=0.8))
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=10)


def plot_residuals_axs(all_data, ax, vlims=[-15, 15]):
    
    asp_dsm_img, _, _, unet_inference_img, _, _, deepdem_inference128_img, _, _, deepdem_inference256_img, _, _, deepdem_inference512_img, _, _, lidar_dsm_img, _, nodata_img, _, _ = all_data
    
    # Plotting residuals
    vmin, vmax = vlims[0], vlims[1]
    cmap = colormaps.get_cmap('RdBu_r')
    cmap.set_bad(color='silver')

    im = ax[0].imshow(compute_residuals(asp_dsm_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
    ax[1].imshow(compute_residuals(unet_inference_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
    ax[2].imshow(compute_residuals(deepdem_inference128_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
    ax[3].imshow(compute_residuals(deepdem_inference256_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
    ax[4].imshow(compute_residuals(deepdem_inference512_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)

    # Adding colorbars
    cax = ax[0].inset_axes([0.03, 0.1, 0.4, 0.03])
    cax.xaxis.set_ticks_position("bottom")
    ax[0].add_patch(Rectangle((0.01, 0.01), 0.45, 0.15, transform=ax[0].transAxes, facecolor="white", alpha=0.8))
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=10)
    
    ax[0].set_ylabel('Residual [m]', fontsize=18)
    ax[5].set_frame_on(False)
    

def plot_ssim_axs(all_data, ax, vlims=[-1, 1]):
    
    _, _, asp_dsm_img_ssim, _, _, unet_inference_img_ssim, _, _, deepdem_inference128_img_ssim, _, _, deepdem_inference256_img_ssim, _, _, deepdem_inference512_img_ssim, lidar_dsm_img, _, nodata_img, _, _ = all_data
    
    # Plotting residuals
    vmin, vmax = vlims[0], vlims[1]
    cmap = colormaps.get_cmap('RdBu_r')
    cmap.set_bad(color='silver')

    im = ax[0].imshow(asp_dsm_img_ssim, vmin=vmin, vmax=vmax, cmap=cmap)
    ax[1].imshow(unet_inference_img_ssim, vmin=vmin, vmax=vmax, cmap=cmap)
    ax[2].imshow(deepdem_inference128_img_ssim, vmin=vmin, vmax=vmax, cmap=cmap)
    ax[3].imshow(deepdem_inference256_img_ssim, vmin=vmin, vmax=vmax, cmap=cmap)
    ax[4].imshow(deepdem_inference512_img_ssim, vmin=vmin, vmax=vmax, cmap=cmap)

    # Adding colorbars
    cax = ax[0].inset_axes([0.03, 0.1, 0.4, 0.03])
    cax.xaxis.set_ticks_position("bottom")
    ax[0].add_patch(Rectangle((0.01, 0.01), 0.45, 0.15, transform=ax[0].transAxes, facecolor="white", alpha=0.8))
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=10)
    
    ax[0].set_ylabel('Hillshade SSIM', fontsize=18)
    ax[5].set_frame_on(False)


# def plot_residuals_axs2(all_data, ax, vlims=[-15, 15]):
    
#     asp_dsm_img, _, unet_inference_img, _, deepdem_inference128_img, _, deepdem_inference256_img, _, deepdem_inference512_img, _, lidar_dsm_img, _, nodata_img, _ = all_data
    
#     # Plotting residuals
#     vmin, vmax = vlims[0], vlims[1]
#     cmap = colormaps.get_cmap('RdBu')
#     cmap.set_bad(color='silver')

#     im = ax[0].imshow(compute_residuals(asp_dsm_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
#     ax[1].imshow(compute_residuals(unet_inference_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
#     ax[2].imshow(compute_residuals(deepdem_inference128_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
#     ax[3].imshow(compute_residuals(deepdem_inference256_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)
#     ax[4].imshow(compute_residuals(deepdem_inference512_img, lidar_dsm_img, nodata_img), vmin=vmin, vmax=vmax, cmap=cmap)

#     # Adding colorbars
#     cax = ax[0].inset_axes([0.03, 0.1, 0.4, 0.03])
#     cax.xaxis.set_ticks_position("bottom")
#     ax[0].add_patch(Rectangle((0.01, 0.01), 0.45, 0.15, transform=ax[0].transAxes, facecolor="white", alpha=0.8))
#     cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
#     cbar.ax.tick_params(labelsize=10)
    
#     ax[0].set_ylabel('Residual [m]', fontsize=18)
#     ax[5].set_frame_on(False)

In [ ]:
add_title = True

fig, ax = plt.subplots(6, 6, figsize=(20, 20))

all_data = read_comparison_files(xmin_list, ymin_list, chipsize, 2)
plot_dsm_axs(all_data, ax[0])
plot_residuals_axs(all_data, ax[1])
plot_ssim_axs(all_data, ax[2])

all_data = read_comparison_files(xmin_list, ymin_list, chipsize, 0)
plot_dsm_axs(all_data, ax[3])
plot_residuals_axs(all_data, ax[4], [-4, 4])
plot_ssim_axs(all_data, ax[5])


if add_title:
    ax[0, 0].set_title('Co-registered stereo', fontsize=17)
    ax[0, 5].set_title('3DEP LiDAR', fontsize=17)
    ax[0, 1].set_title('Baker 2015 -> Baker 2015', fontsize=17)
    ax[0, 2].set_title('Baker 2015 → Baker 2013', fontsize=17)
    ax[0, 3].set_title('Baker 2015-> Baker 2017', fontsize=17)
    ax[0, 4].set_title('CO 2019-> Baker 2015', fontsize=17)

for ax_ in ax.flat:
    ax_.set_xticks([])
    ax_.set_yticks([])

plt.tight_layout()
plt.savefig(out_figpath / 'baker_transfer_comparison.png', dpi=300, bbox_inches='tight')

### Table 4

In [ ]:
for _n in ['all_data']:
    globals().pop(_n, None)
gc.collect()

file_list = [output_data_path / inf_file(k) for k in ['wv2013', 'lt256', 'wv2017', 'co', 'baker_co']]
labels = ['2013', '2015', '2017', 'COWts', 'BakerCO']
df_val, _, class_info, _, _ = fu.compute_DSM_stats(file_list, labels, input_data_path, train_split, worldcover_path, compute_train=False)
df_val

## 7. DTM prediction (Fig. 9, Table 5)

Same figure code as Fig. 6, driven with the lidar DTM and the DTM-trained model. Only the column titles
(DSM -> DTM) and the output file name differ from the Fig. 6 call.

In [ ]:
# back to the Fig. 6 tile windows, DTM reference rasters
_w = list(cfg['windows'].values())
xmin_list = [w['xmin'] for w in _w]
ymin_list = [w['ymax'] for w in _w]
chipsize = [w['size'] for w in _w]

nodata_mask_path = 'final_nodata_dtm_mask.tif'
lidar_data_path = 'final_lidar_dtm_data.tif'
lidar_hs_data_path = 'final_lidar_dtm_data_hs_multi.tif'

In [ ]:
# tile reader with the DTM reference rasters; model slots: Baker 2013, Baker 2017, DeepDEM DTM, Colorado -> Baker
read_comparison_files = functools.partial(
    fu.read_comparison_files, input_data_path=input_data_path, output_data_path=output_data_path,
    models=[inf_file(k) for k in ['wv2013', 'wv2017', 'dtm', 'co']],
    lidar_data_path=lidar_data_path, lidar_hs_data_path=lidar_hs_data_path, nodata_mask_path=nodata_mask_path)

In [ ]:
fig, ax = plt.subplots(3, 5, figsize = (17.8, 10.5))
plt.subplots_adjust(wspace=0.01, hspace=0.04)

all_data1 = read_comparison_files(xmin_list, ymin_list, chipsize, 0)
ssim_asp1, ssim_deepdem1 = plot_dsm_visual(all_data1, chipsize[0], fig, ax[0], xlims=[-3, 3], no_label='1) Bare/Sparse Vegetation ')
all_data2 = read_comparison_files(xmin_list, ymin_list, chipsize, 1)
ssim_asp2, ssim_deepdem2 = plot_dsm_visual(all_data2, chipsize[1], fig, ax[1], xlims=[-2, 2], no_label='2) Snow and Ice                ')
all_data3 = read_comparison_files(xmin_list, ymin_list, chipsize, 2)
ssim_asp3, ssim_deepdem3 = plot_dsm_visual(all_data3, chipsize[2], fig, ax[2], xlims=[-16, 40], no_label='3) Dense Vegetation         ', label_side='right')

ax[0, 0].set_title('Reference ortho image', fontsize=13)
ax[0, 1].set_title('Co-registered stereo DSM', fontsize=13)
ax[0, 2].set_title('DeepDEM DTM', fontsize=13)
ax[0, 3].set_title('3DEP LiDAR DTM', fontsize=13)
ax[0, 4].set_title('Residual histograms [m]', fontsize=13)

# Turn off tickmarks for images
for ax_ in ax[:, 0:4].flat:
    ax_.set_xticklabels('')
    ax_.set_yticklabels('')
    ax_.set(xticks=[], yticks=[])
    
plt.savefig(out_figpath / 'baker_dtm_comparison.png', dpi=300, bbox_inches='tight')

### Table 5

In [ ]:
for _n in ['all_data1', 'all_data2', 'all_data3', 'ssim_asp1', 'ssim_asp2', 'ssim_asp3', 'ssim_deepdem1', 'ssim_deepdem2', 'ssim_deepdem3']:
    globals().pop(_n, None)
gc.collect()

file_list = [input_data_path / 'final_asp_dsm.tif', output_data_path / inf_file('dtm')]
labels = ['ASP', 'DeepDTM']
df_val, _, class_info, _, _ = fu.compute_DSM_stats(file_list, labels, input_data_path, train_split, worldcover_path,
                                                   lidar_path=lidar_data_path, nodata_path=nodata_mask_path, compute_train=False)

# restore the DSM reference rasters
nodata_mask_path = 'final_nodata_mask.tif'
lidar_data_path = 'final_lidar_data.tif'
lidar_hs_data_path = 'final_lidar_data_hs_multi.tif'

df_val